# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum / portfolio',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'services / skills', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio project',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio project', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 2 relevant links


{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community page', 'url': 'https://discuss.huggingface.co'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
GGML and llama.cpp join Hugging Face 🔥
Try HuggingChat Omni – Chat with AI 💬
Get started with Inference in seconds 🚀
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-397B-A17B
Updated
1 day ago
•
390k
•
981
zai-org/GLM-5
Updated
11 days ago
•
180k
•
1.5k
Nanbeige/Nanbeige4.1-3B
Updated
3 days ago
•
202k
•
768
nvidia/personaplex-7b-v1
Updated
9 days ago
•
539k
•
2.18k
MiniMaxAI/MiniMax-M2.5
Updated
8 days ago
•
224k
•
899
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.59k
Qwen Image Multiple Angles 3D Camera
🎥
1.59k
Change the camera angle of a photo with AI
Run

In [1]:
#brochure_system_prompt = """
#You are an assistant that analyzes the contents of several relevant pages from a company website
#and creates a short brochure about the company for prospective customers, investors and recruits.
#Respond in markdown without code blocks.
#Include details of company culture, customers and careers/jobs if you have the information.
#"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
 You are an assistant that analyzes the contents of several relevant pages from a company website
 and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
 Respond in markdown without code blocks.
 Include details of company culture, customers and careers/jobs if you have the information.
"""


## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face Brochure

---

## Who We Are

Hugging Face is the AI community **building the future** of machine learning. We are a collaboration platform empowering the global machine learning community to create, share, discover, and experiment with models, datasets, and applications. Known as the **home of machine learning**, Hugging Face brings together engineers, scientists, and AI enthusiasts to contribute to and learn from the world’s largest open-source ML ecosystem.

---

## Our Mission

At Hugging Face, our mission is to **democratize good machine learning, one commit at a time**. We enable the next generation of machine learning engineers and scientists to collaborate openly and ethically, driving the AI revolution forward in an accessible and inclusive manner.

---

## What We Offer

- **Model Hub:** Browse and collaborate on over **2 million** pre-trained models across multiple modalities including text, image, video, audio, and 3D.  
- **Datasets:** Access and contribute to a collection of over **500,000 datasets** to accelerate your machine learning projects.  
- **Spaces:** Host and share interactive AI applications and demos; explore 1 million+ AI applications created by the community.  
- **Inference & Tools:** Get started with inference in seconds through our open-source stack and APIs.  
- **Community:** Engage with a rapidly growing community of over **83,000 members**, sharing knowledge, updates, and innovation in the ML space.  

---

## Company Culture

- **Open & Collaborative:** We believe in transparency and shared contributions. Everyone is invited to co-create and collaborate on open-source AI tools and resources.  
- **Inclusive & Ethical AI:** We empower developers and researchers globally to build an open and ethical AI future.  
- **Learning & Growth:** Offering resources like Hugging Face Fundamentals and a rich ecosystem to grow skills, knowledge, and influence in AI.  
- **Passionate & Innovative:** Our team of 187+ talented individuals includes experts pushing the boundaries of AI technology and research.

---

## Our Customers & Community

Hugging Face serves a wide variety of users:
- Individual ML engineers and data scientists building cutting-edge AI applications.  
- Organizations looking to accelerate AI adoption and innovation through open-source tools and enterprise solutions.  
- Researchers aiming to share datasets and models with the global community.  
- ML enthusiasts and learners eager to grow through collaborative projects and learning tracks.

Our platform supports millions of ML models and datasets updated regularly by both our science team and the community.

---

## Career Opportunities

If you are driven to **democratize machine learning** and want to contribute to one of the fastest-growing AI ecosystems, Hugging Face offers exciting career opportunities in areas like:
- Machine learning research and engineering  
- Software development for open-source projects  
- Community engagement and developer relations  
- Data science and AI operations  
- Product management and enterprise solutions  

Join a dynamic and passionate team dedicated to building the future of AI. Visit our [Careers page](https://huggingface.co/careers) to explore openings and be part of a mission-driven company.

---

## Contact & Connect

- Website: [https://huggingface.co](https://huggingface.co)  
- Community Forum and Discussions: Join our Discord and GitHub communities  
- Social Channels: Twitter, LinkedIn for latest updates and engagement  
- Press Inquiries and Partnerships: Contact via email on our website  

---

## Color Palette & Brand

- **Signature Colors:**  
  - Yellow: #FFD21E  
  - Orange: #FF9D00  
  - Gray: #6B7280  

Hugging Face brand assets and logos are freely available to align with our open and collaborative philosophy.

---

Experience the power of collaboration in AI with Hugging Face — the platform where **machine learning communities unite to build the future.**  
Sign up today and join millions accelerating AI innovation globally!

In [21]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the AI community building the future of machine learning. As a collaboration platform, Hugging Face empowers machine learning engineers, scientists, and AI enthusiasts to create, discover, and share models, datasets, and applications. Through its open and ethical AI mission, the company brings together a fast-growing global community dedicated to advancing AI technologies responsibly.

---

## What We Offer

### The Hugging Face Hub
- **2M+ Models:** Browse through millions of state-of-the-art machine learning models covering text, image, video, audio, and even 3D data.
- **500k+ Datasets:** Access diverse datasets curated for different machine learning tasks.
- **1M+ Applications:** Explore and build AI applications using pre-trained models and datasets.
- **Spaces:** Host and collaborate on interactive machine learning demos and apps with community support.

### Open Source Innovation
Utilize Hugging Face's open-source stack to accelerate ML workflows. The platform supports hosting unlimited public models, datasets, and applications, helping users move faster from concept to deployment.

---

## Company Culture

- **Community-Driven:** Hugging Face fosters an inclusive, vibrant community where collaboration and openness are central values.
- **Ethical AI Commitment:** The company is dedicated to building AI technologies that are transparent, responsible, and beneficial for everyone.
- **Learning & Sharing:** The platform enables users to build their machine learning portfolios by sharing and showcasing their work to the world.
- **Innovation at Scale:** Hugging Face supports exploring all AI modalities, including text, image, video, audio, and 3D groundbreaking research.

---

## Our Customers & Community

- **Machine Learning Engineers & Researchers:** Empowered by access to the latest pre-trained models and datasets.
- **Enterprises:** Leveraging scalable and customizable AI solutions through the Hugging Face Enterprise platform.
- **Developers & AI Enthusiasts:** Building applications and contributing to the open-source ecosystem.
- **Academic Institutions:** Collaborating and sharing cutting-edge AI research and datasets.

---

## Careers at Hugging Face

Join a community passionate about advancing the future of AI. Hugging Face offers exciting opportunities to work with cutting-edge technologies in a collaborative and ethical environment. The team is continuously growing and looks for talent in areas including:

- Machine Learning Engineering
- Data Science
- Software Development
- Research
- Community & Developer Relations

**Why work at Hugging Face?**

- Be part of a global community pushing the boundaries of open-source AI.
- Shape the ethical frameworks that guide AI development.
- Collaborate with world-renowned experts in a supportive environment.
- Enjoy a culture that values transparency, learning, and innovation.

---

## How to Get Started

- Explore and contribute to millions of open-source ML models and datasets at [huggingface.co](https://huggingface.co).
- Try the latest AI applications like HuggingChat Omni to experience chat-based AI.
- Sign up to create your ML portfolio and host your projects on the Hugging Face Hub.
- Join the community discussions and collaborate on innovative AI projects.

---

## Brand Colors & Assets

- **Primary Colors:** 
  - Yellow #FFD21E
  - Orange #FF9D00
  - Gray #6B7280

- Professional and open brand identity with logos available in SVG, PNG, and AI formats.

---

Hugging Face is more than a platform—it's a movement towards an open, collaborative, and ethical AI future.

**Join us today at [huggingface.co](https://huggingface.co).**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>